# Solution — Multi-API Kafka Pipeline: Producers

Two producers in this notebook:
- **Weather Producer** — polls Open-Meteo every 30 seconds, one message per city → `weather` topic
- **Bikes Producer** — polls the Vélib bike-share API every 60 seconds, one message per station → `bikes` topic

> No API key required for either API.

---
## Shared Setup

In [ ]:
from kafka import KafkaProducer
from datetime import datetime
from time import sleep
import requests
import json

BROKERS = ['course-kafka:9092']

---
## Producer 1 — Weather (`weather` topic)

Uses the [Open-Meteo API](https://open-meteo.com) — completely free, no API key needed.

In [ ]:
WEATHER_TOPIC = 'weather'

CITIES = [
    {"name": "Tel Aviv",  "lat": 32.07,  "lon": 34.78},
    {"name": "London",    "lat": 51.51,  "lon": -0.13},
    {"name": "New York",  "lat": 40.71,  "lon": -74.01},
]

# Map Open-Meteo numeric weather codes to human-readable strings.
# Full list: https://open-meteo.com/en/docs#weathervariables
WEATHER_CODES = {
    0:  "Clear sky",
    1:  "Mainly clear",
    2:  "Partly cloudy",
    3:  "Overcast",
    61: "Rain",
    80: "Showers"
}

In [ ]:
weather_producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

In [ ]:
while True:
    for city in CITIES:

        # Fetch current weather from Open-Meteo — no API key required.
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude":        city["lat"],
                "longitude":       city["lon"],
                "current_weather": True,
                "wind_speed_unit": "kmh"
            },
            timeout=10
        )
        cw = response.json()["current_weather"]

        # Build the enriched payload.
        # WEATHER_CODES.get(code, "Unknown") handles codes not in our lookup table gracefully.
        payload = {
            "city":        city["name"],
            "latitude":    city["lat"],
            "longitude":   city["lon"],
            "temperature": cw["temperature"],
            "windspeed":   cw["windspeed"],
            "condition":   WEATHER_CODES.get(cw["weathercode"], "Unknown"),
            "timestamp":   datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        # Use city name as key so messages for the same city always land
        # in the same partition — useful for per-city ordered processing downstream.
        weather_producer.send(
            topic = WEATHER_TOPIC,
            key   = city["name"].encode("utf-8"),
            value = json.dumps(payload).encode("utf-8")
        )
        weather_producer.flush()

        print(f"Sent → {city['name']} | {payload['temperature']}°C | {payload['condition']}")

    print(f"--- cycle done, sleeping 30s ---")
    sleep(30)

---
## Producer 2 — Bike-share stations (`bikes` topic)

Uses the open [Vélib Métropole GBFS feed](https://www.velib-metropole.fr/donnees-open-data-gbfs-du-service-velib-metropole) (Paris bike-share) — no API key needed.

Business story: an operations team moves bikes between stations by truck, so they need a live stream of how many bikes and free docks each station has.

In [ ]:
BIKES_TOPIC = 'bikes'

INFO_URL   = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_information.json"
STATUS_URL = "https://velib-metropole-opendata.smovengo.cloud/opendata/Velib_Metropole/station_status.json"

STATIONS_TO_TRACK = 5   # keep the stream small and readable in class

In [ ]:
# Station names and capacity rarely change, so we fetch them ONCE before the loop
# and keep them in a lookup dict: { station_id: {"name": ..., "capacity": ...} }.
# Later, every live status message is ENRICHED with the name/capacity from this dict.
info = requests.get(INFO_URL, timeout=10).json()["data"]["stations"][:STATIONS_TO_TRACK]
stations = {s["station_id"]: {"name": s["name"], "capacity": s["capacity"]} for s in info}

for sid, s in stations.items():
    print(f"Tracking {sid}: {s['name']} (capacity {s['capacity']})")

In [ ]:
bikes_producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

In [ ]:
while True:

    # One call returns the live status of ALL stations (~1,500) — we keep only the ones we track.
    response = requests.get(STATUS_URL, timeout=10)
    all_status = response.json()["data"]["stations"]

    for st in all_status:
        sid = st["station_id"]
        if sid not in stations:
            continue    # not one of our 5 stations — skip it

        payload = {
            "station_id":      sid,
            "name":            stations[sid]["name"],       # enrichment from the lookup dict
            "capacity":        stations[sid]["capacity"],
            "bikes_available": st["num_bikes_available"],
            "docks_available": st["num_docks_available"],
            "timestamp":       datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

        # Use station_id as key -> all updates of one station go to the same partition, in order.
        # That ordering matters: the consumer compares each update to the PREVIOUS one.
        # station_id is an int, so convert to str first, then to bytes.
        bikes_producer.send(
            topic = BIKES_TOPIC,
            key   = str(sid).encode("utf-8"),
            value = json.dumps(payload).encode("utf-8")
        )
        bikes_producer.flush()

        print(f"Sent → {payload['name']}: {payload['bikes_available']} bikes, {payload['docks_available']} free docks")

    print("--- cycle done, sleeping 60s ---")
    sleep(60)